A dataset generator.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr


In [16]:
MODEL_GPT = 'gpt-4.1-mini'
MODEL_LLAMA = 'llama3.2:1b'




In [17]:
load_dotenv(override=True)
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openrouter_api_key:
    print(f"Openrouter API Key exists and begins {openrouter_api_key[:8]}")
else:
    print("Openrouter API Key not set")
    

Openrouter API Key exists and begins sk-or-v1


In [18]:
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OLLAMA_BASE_URL = "http://localhost:11434/v1"

openrouter = OpenAI(api_key=openrouter_api_key, base_url=OPENROUTER_BASE_URL)
ollama = OpenAI(api_key="ollama", base_url=OLLAMA_BASE_URL)


In [23]:
clients = {"gpt": openrouter, "llama3.2:1b": ollama, }

In [32]:
def stream_gpt(history, message):
    relevant_system_message = """
    You are a helpful assistant for generating synthetic dataset. Always generate a JSON dataset.
    For example: Generate a dataset for a Product entity
    [{
        "title": "Wireless Mouse",
        "price": 29.99,
        "category": "Electronics",
        "in_stock": true
    }]
    """
    messages = [{"role": "system", "content": relevant_system_message}] + history + [{"role": "user", "content": message}]

    stream = openrouter.chat.completions.create(
        model=MODEL_GPT,
        messages=messages,
        stream=True,
        max_tokens=4096,
    )
    result = ""
    for chunk in stream:
        if chunk.choices:
            result += chunk.choices[0].delta.content or ""
        yield result

In [21]:
def stream_ollama(history, message):
    relevant_system_message = """
     You are a helpful assistant for generating synthetic dataset. Always generate a JSON dataset.
    For example: Generate a dataset for a Product entity
    [{
        "title": "Wireless Mouse",
        "price": 29.99,
        "category": "Electronics",
        "in_stock": true
    }]
    """
    
    messages = [{"role": "system", "content": relevant_system_message}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model=MODEL_LLAMA, messages=messages)
    print(response)

    yield response.choices[0].message.content

In [33]:
STREAM_FUNCTIONS = {
    "GPT": stream_gpt,
    "LLAMA3.2:1B": stream_ollama,
}

def run_chat(msg, _chatbot):
    """Add user message to display and pass context for API call. Clears previous history for each new message."""
    updated = [{"role": "user", "content": msg}]  # Fresh display: only current message
    return "", updated, ([], msg)  # Empty history for API - each message is independent

def stream_to_chatbot(context, model):
    """Stream API response based on selected model and yield message history for Chatbot."""
    history, message = context
    stream_fn = STREAM_FUNCTIONS.get(model, stream_gpt)
    for chunk in stream_fn(history, message):
        yield history + [{"role": "user", "content": message}, {"role": "assistant", "content": chunk}]

# UI definition
COL_HEIGHT = 450

with gr.Blocks() as ui:
    chat_context = gr.State(value=([], ""))
    with gr.Row(elem_id="columns-row"):
        with gr.Column(scale=1, elem_classes=["scrollable-column"]):
            message = gr.Textbox(
                label="Message",
                lines=15,
                max_lines=20,
                placeholder="Please provide the entity/schema you want to generate a dataset for..."
            )
        with gr.Column(scale=1, elem_classes=["scrollable-column"]):
            chatbot = gr.Chatbot(
                height=COL_HEIGHT,
                type="messages",
            )
    with gr.Row():
        model_dropdown = gr.Dropdown(
            choices=["GPT", "LLAMA3.2:1B"],
            value="GPT",
            label="Select Model",
        )
    with gr.Row():
        submit_btn = gr.Button("Generate Dataset")

    # Hooking up events to callbacks
    submit_btn.click(run_chat, [message, chatbot], [message, chatbot, chat_context]).then(
        stream_to_chatbot, [chat_context, model_dropdown], chatbot
    )
    message.submit(run_chat, [message, chatbot], [message, chatbot, chat_context]).then(
        stream_to_chatbot, [chat_context, model_dropdown], chatbot
    )

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


ChatCompletion(id='chatcmpl-825', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Here is a JSON dataset for a "Product" entity:\n\n```json\n{\n  "Products": [\n    {\n      "id": 1,\n      "title": "Wireless Mouse",\n      "price": 29.99,\n      "category": "Electronics",\n      "in_stock": true\n    },\n    {\n      "id": 2,\n      "title": "Printers",\n      "price": 49.95,\n      "category": "Office Supplies",\n      "in_stock": false\n    },\n    {\n      "id": 3,\n      "title": "Scanner",\n      "price": 69.99,\n      "category": "Electronics",\n      "in_stock": true\n    },\n    {\n      "id": 4,\n      "title": "Laptop", \n      "price": 999.00,  \n      "category": "Computer Accessories",\n      "in_stock": false\n    }\n  ]\n}\n```\n\nYou can use this dataset in your applications by simply fetching and manipulating the JSON data as you see fit. Let me know if you have any further requests!', refusal=None, role='assistant'